# Data Analysis Using DuckDB

Using DuckDB for local analytics is a strong choice when we want fast, zero-infrastructure querying directly on files (CSV, Parquet, JSON) without spinning up a database server.

DuckDB is an **in-process OLAP engine** (like SQLite for analytics). It runs inside your app (Python, CLI, etc.) and queries files **in-place** using vectorized execution. DuckDB gives you **warehouse-grade analytics locally**, with near-zero setup. It’s especially powerful for engineers working with **data files, pipelines, or exploratory analysis**.

Use it when:

- You need local analytics without infra
- You’re working with large files (GBs)
- You want SQL over data lake files
- You need fast prototyping / exploration

Avoid if:

- You need high-concurrency OLTP
- You need distributed cluster scaling (use Spark/BigQuery)
- You need multi-user transactional systems



In [12]:
import duckdb

with duckdb.connect('databases/hotels.duckdb') as conn:
    conn.sql("DROP TABLE IF EXISTS bookings;")
    conn.sql("""
    CREATE TABLE IF NOT EXISTS bookings (
        date DATE,
        room_type STRING,
        price FLOAT,
        breakfast BOOLEAN,
        name STRING,
        email STRING,
        guests UINT8
    );
    """)

    conn.sql("""
    INSERT INTO bookings VALUES
        ('2024-01-15', 'Single', 150.00, True, 'John Doe', 'john@email.com', 2),
        ('2024-01-16', 'Double', 220.00, False, 'Jane Smith', 'jane@email.com', 3),
        ('2024-01-17', 'Suite', 450.00, True, 'Bob Johnson', 'bob@email.com', 1),
        ('2024-01-18', 'Single', 150.00, True, 'Alice Brown', 'alice@email.com', 1),
        ('2024-01-19', 'Double', 220.00, True, 'Charlie Davis', 'charlie@email.com', 4),
        ('2024-01-20', 'Suite', 450.00, False, 'Eva Wilson', 'eva@email.com', 2),
        ('2024-01-21', 'Single', 150.00, False, 'Frank Miller', 'frank@email.com', 1),
        ('2024-01-22', 'Double', 220.00, True, 'Grace Lee', 'grace@email.com', 3);
    """)

    print(conn.sql("SELECT * FROM bookings"))
    print(f"{type(conn.sql('FROM bookings;')) = }")
    bookings = conn.sql("FROM bookings;").df()

bookings


┌────────────┬───────────┬───────┬───────────┬───────────────┬───────────────────┬────────┐
│    date    │ room_type │ price │ breakfast │     name      │       email       │ guests │
│    date    │  varchar  │ float │  boolean  │    varchar    │      varchar      │ uint8  │
├────────────┼───────────┼───────┼───────────┼───────────────┼───────────────────┼────────┤
│ 2024-01-15 │ Single    │ 150.0 │ true      │ John Doe      │ john@email.com    │      2 │
│ 2024-01-16 │ Double    │ 220.0 │ false     │ Jane Smith    │ jane@email.com    │      3 │
│ 2024-01-17 │ Suite     │ 450.0 │ true      │ Bob Johnson   │ bob@email.com     │      1 │
│ 2024-01-18 │ Single    │ 150.0 │ true      │ Alice Brown   │ alice@email.com   │      1 │
│ 2024-01-19 │ Double    │ 220.0 │ true      │ Charlie Davis │ charlie@email.com │      4 │
│ 2024-01-20 │ Suite     │ 450.0 │ false     │ Eva Wilson    │ eva@email.com     │      2 │
│ 2024-01-21 │ Single    │ 150.0 │ false     │ Frank Miller  │ frank@email.com  

,date,room_type,price,breakfast,name,email,guests
0,2024-01-15,Single,150.0,True,John Doe,john@email.com,2
1,2024-01-16,Double,220.0,False,Jane Smith,jane@email.com,3
2,2024-01-17,Suite,450.0,True,Bob Johnson,bob@email.com,1
3,2024-01-18,Single,150.0,True,Alice Brown,alice@email.com,1
4,2024-01-19,Double,220.0,True,Charlie Davis,charlie@email.com,4
5,2024-01-20,Suite,450.0,False,Eva Wilson,eva@email.com,2
6,2024-01-21,Single,150.0,False,Frank Miller,frank@email.com,1
7,2024-01-22,Double,220.0,True,Grace Lee,grace@email.com,3


## Read CSV Files

In [13]:
invoice = duckdb.sql("""
    SELECT *
    FROM 'data/Leverantorsfaktura202408.csv';
""").df()

print(f"{invoice.shape = }")
invoice.head()


invoice.shape = (92989, 7)


,Förvaltning,Leverantör,Organisationsnummer,Verifikationsnummer,Konto,Kontotext,Belopp exkl moms
0,Stadsmiljönämnden,TRACK TEC GMBH,106/5727/0626,4001291513,4101,Inköp anläggnings och underhållsmaterial,"9 835 315,00"
1,Kretslopp och Vatten,POLISMYNDIGHETEN I VÄSTRA GÖTALAND,2021000076,5601378982,6185,Anläggningsentreprenad,"870,00"
2,Kretslopp och Vatten,POLISMYNDIGHETEN I VÄSTRA GÖTALAND,2021000076,5601377374,6185,Anläggningsentreprenad,"870,00"
3,Kretslopp och Vatten,POLISMYNDIGHETEN I VÄSTRA GÖTALAND,2021000076,5601378519,6185,Anläggningsentreprenad,"870,00"
4,Exploateringsnämnden,POLISMYNDIGHETEN I VÄSTRA GÖTALAND,2021000076,2001226894,7641,Diverse skatter och offentliga avgifter,"1 000,00"


In [14]:
finance = duckdb.sql("""
    SELECT *
    FROM 'data/financial_data.csv';
""").df()

finance.head()


,transaction_id,date,company,transaction_type,category,amount,currency,account_number,description,status,payment_method,tax_amount,net_amount
0,TXN1000,2024-01-15,UnitedHealth Group,Investment,Marketing,498789.60,JPY,ACC90616,Q1 revenue for r&d,Completed,Check,15640.11,483149.49
1,TXN1001,2024-12-22,Broadcom Inc,Expense,Administrative,407890.44,GBP,ACC72475,Q4 capital gain for it,Completed,Wire Transfer,25883.85,382006.59
2,TXN1002,2024-10-11,Starbucks,Capital Gain,Operations,363927.61,USD,ACC26222,Q3 expense for administrative,Pending,ACH,24814.56,339113.05
3,TXN1003,2024-09-30,Lockheed Martin,Dividend,R&D,336378.01,EUR,ACC86805,Q4 investment for marketing,Completed,Check,46359.17,290018.84
4,TXN1004,2024-07-02,IBM Corp,Operating Cost,Administrative,193284.51,GBP,ACC72617,Q2 expense for distribution,Completed,Credit Card,41529.35,151755.16


## Read CSV Files and Combine them

In [15]:
bookings = duckdb.sql("""
    SELECT * FROM 'data/hotel*.csv'
""").df()

bookings


,booking_id,guest_name,check_in,check_out,room_type,guests,price_sek_per_night,breakfast,source,status
0,H2025-01-001,Anna Berg,2025-01-05,2025-01-07,Standard,2,1150,True,Direct,Confirmed
1,H2025-01-002,Johan Nilsson,2025-01-10,2025-01-11,Single,1,890,False,Booking.com,Confirmed
2,H2025-01-003,Sofia Lind,2025-01-14,2025-01-16,Deluxe,2,1450,True,Expedia,Confirmed
3,H2025-01-004,Erik Svensson,2025-01-20,2025-01-22,Standard,3,1250,True,Direct,Confirmed
4,H2025-01-005,Maria Johansson,2025-01-27,2025-01-28,Single,1,920,False,Hotels.com,Cancelled
5,H2025-02-006,Daniel Karlsson,2025-02-02,2025-02-04,Standard,2,1190,True,Direct,Confirmed
6,H2025-02-007,Elin Andersson,2025-02-07,2025-02-09,Deluxe,2,1520,True,Booking.com,Confirmed
7,H2025-02-008,Lukas Eriksson,2025-02-12,2025-02-13,Single,1,940,False,Expedia,Confirmed
8,H2025-02-009,Olivia Larsson,2025-02-18,2025-02-20,Standard,2,1210,True,Direct,Confirmed
9,H2025-02-010,Viktor Persson,2025-02-24,2025-02-26,Deluxe,3,1580,True,Hotels.com,Confirmed


## Read JSON Data

Problem:
- With a nested data structure, it can be hard to find the data you need.

In [16]:
duckdb.sql("""
    SELECT *
    FROM 'data/library.json'
""").df()


,name,books
0,Coolu Libraru,"[{'id': 1, 'title': 'The Hitchhiker's Guide to..."


### Unnesting JSON Data

When we want to extract specific columns from a nested data structure, we can use the UNNEST function.

In [17]:
duckdb.sql("""
    SELECT 
        l.name as library_name,
        UNNEST(l.books, max_depth := 2)
    FROM 'data/library.json' AS l 
""").df()


,library_name,id,title,author,year
0,Coolu Libraru,1,The Hitchhiker's Guide to the Galaxy,Douglas Adams,1979
1,Coolu Libraru,2,Pride and Prejudice,Jane Austen,1813
2,Coolu Libraru,3,1984,George Orwell,1949
3,Coolu Libraru,4,To Kill a Mockingbird,Harper Lee,1960
4,Coolu Libraru,5,The Great Gatsby,F. Scott Fitzgerald,1925
5,Coolu Libraru,6,Moby Dick,Herman Melville,1851
6,Coolu Libraru,7,War and Peace,Leo Tolstoy,1869
7,Coolu Libraru,8,The Lord of the Rings,J.R.R. Tolkien,1954
8,Coolu Libraru,9,Crime and Punishment,Fyodor Dostoevsky,1866
9,Coolu Libraru,10,Don Quixote,Miguel de Cervantes,1605


### Using Cross Joins

- It does cartesian product between the name and each entry in the unnested list
- Basically performing a for-loop in DuckDB

In [18]:
duckdb.sql("""
    SELECT 
           l.name, 
           b
    FROM 'data/library.json' l
    CROSS JOIN UNNEST(l.books) b
""").df()


,name,b
0,Coolu Libraru,"{'unnest': {'id': 1, 'title': 'The Hitchhiker'..."
1,Coolu Libraru,"{'unnest': {'id': 2, 'title': 'Pride and Preju..."
2,Coolu Libraru,"{'unnest': {'id': 3, 'title': '1984', 'author'..."
3,Coolu Libraru,"{'unnest': {'id': 4, 'title': 'To Kill a Mocki..."
4,Coolu Libraru,"{'unnest': {'id': 5, 'title': 'The Great Gatsb..."
5,Coolu Libraru,"{'unnest': {'id': 6, 'title': 'Moby Dick', 'au..."
6,Coolu Libraru,"{'unnest': {'id': 7, 'title': 'War and Peace',..."
7,Coolu Libraru,"{'unnest': {'id': 8, 'title': 'The Lord of the..."
8,Coolu Libraru,"{'unnest': {'id': 9, 'title': 'Crime and Punis..."
9,Coolu Libraru,"{'unnest': {'id': 10, 'title': 'Don Quixote', ..."


and extract the columns we want:

In [19]:
books = duckdb.sql("""
    SELECT 
           l.name, 
           b.unnest.title,
           b.unnest.author,
           b.unnest.year,
    FROM 'data/library.json' l
    CROSS JOIN UNNEST(l.books) b
""").df()

books


,name,title,author,year
0,Coolu Libraru,The Hitchhiker's Guide to the Galaxy,Douglas Adams,1979
1,Coolu Libraru,Pride and Prejudice,Jane Austen,1813
2,Coolu Libraru,1984,George Orwell,1949
3,Coolu Libraru,To Kill a Mockingbird,Harper Lee,1960
4,Coolu Libraru,The Great Gatsby,F. Scott Fitzgerald,1925
5,Coolu Libraru,Moby Dick,Herman Melville,1851
6,Coolu Libraru,War and Peace,Leo Tolstoy,1869
7,Coolu Libraru,The Lord of the Rings,J.R.R. Tolkien,1954
8,Coolu Libraru,Crime and Punishment,Fyodor Dostoevsky,1866
9,Coolu Libraru,Don Quixote,Miguel de Cervantes,1605


## Export to CSV

In [20]:
books.to_csv("data/books.csv")